In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
from tensorflow.keras.models import load_model
import os

In [2]:
# 1. Define Paths
MODEL_PATH = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Best_Models\fold_4_best.h5"
AUTISTIC_DIR = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\step4\Autistic"
NON_AUTISTIC_DIR = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\step4\Non_Autistic"

In [3]:
print("Loading model... please wait.")
model = load_model(MODEL_PATH)
print("Model loaded successfully!")

Loading model... please wait.
Model loaded successfully!


In [4]:
def create_class_dataframe(directory, label_name, label_value):
    # Get list of all image files
    files = [f for f in os.listdir(directory) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    # Create the DataFrame
    df = pd.DataFrame({
        'File_Name': files,
        'Full_Path': [os.path.join(directory, f) for f in files],
        'Class': label_name,
        'Label': label_value  # 0 for Autistic, 1 for Non-Autistic
    })
    return df

In [5]:
# 3. Create the two DataFrames using the corrected variable names
df_autistic = create_class_dataframe(AUTISTIC_DIR, "Autistic", 1)
df_non_autistic = create_class_dataframe(NON_AUTISTIC_DIR, "Non_Autistic", 0)

In [6]:
# 4. Display counts to verify success
print(f"Successfully created df_autistic with {len(df_autistic)} images.")
print(f"Successfully created df_non_autistic with {len(df_non_autistic)} images.")

Successfully created df_autistic with 1952 images.
Successfully created df_non_autistic with 1965 images.


In [7]:
df_master = pd.concat([df_autistic, df_non_autistic], ignore_index=True)
print(f"Total master dataframe size: {len(df_master)}")

print("Value counts for Label:")
print(df_master['Label'].value_counts())


print("\nSample of combined data:")
display(df_master.sample(5))

Total master dataframe size: 3917
Value counts for Label:
Label
0    1965
1    1952
Name: count, dtype: int64

Sample of combined data:


,File_Name,Full_Path,Class,Label
2365,191.jpg,C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Researc...,Non_Autistic,0
860,463.jpg,C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Researc...,Autistic,1
2962,525.jpg,C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Researc...,Non_Autistic,0
520,157.jpg,C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Researc...,Autistic,1
2446,1983.jpg,C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Researc...,Non_Autistic,0


In [15]:
def extract_features_to_dataframe(df, model):
    all_features = []
    print("extration strat")
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = row['Full_Path']
        
        try:
            # --- 1. PREPROCESSING ---
            img = tf.io.read_file(img_path)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.resize(img, [224, 224])
            img = img[..., ::-1] # RGB to BGR
            mean = [93.5940, 104.7624, 129.1863]
            img = img - mean
            img_tensor = tf.expand_dims(img, 0)

            # --- 2. CAPTURING BOTH OUTPUTS ---
            # Your model returns: [classification_output, asd_feature_vector]
            prediction, feature_vector = model.predict(img_tensor, verbose=0)
            
            # Extract Probability (Output 0)
            prob_score = float(prediction[0][0])
            
            # Extract 256-D Vector (Output 1)
            vec_256 = feature_vector[0]

            # --- 3. SAVING TO DATA ROW ---
            record = {
                'File_Name': row['File_Name'],
                'Actual_Label': row['Label'],   # 1=Autistic, 0=Non_Autistic
                'Model_Prob': prob_score,       # <--- OUTPUT 1 (Probability)
                'Model_Pred': 1 if prob_score > 0.1 else 0 # <--- OUTPUT 1 (Thresholded)
            }
            
            # Add Vector Columns (Output 2)
            for i, val in enumerate(vec_256):
                record[f'V_{i}'] = val          # <--- OUTPUT 2 (Vector)
                
            all_features.append(record)
            
        except Exception as e:
            print(f"Error in {row['File_Name']}: {e}")

    print("extractiion done")
    return pd.DataFrame(all_features)

In [16]:
print("Extracting Features... please wait.")
df_final_results = extract_features_to_dataframe(df_master, model)
print(" Features Extracted.......... :)")

Extracting Features... please wait.
extration strat


100%|██████████████████████████████████████████████████████████████████████████████| 3917/3917 [07:29<00:00,  8.72it/s]


extractiion done
 Features Extracted.......... :)


In [18]:
# Save to your desktop
save_path = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\Extracted features\ImgFeatures.csv"
df_final_results.to_csv(save_path, index=False)

print(f"\nSuccessfully saved {len(df_final_results)} rows to {save_path}")


Successfully saved 3917 rows to C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\Extracted features\ImgFeatures.csv
